In [1]:
# Install specialized football and ML libraries
!pip install statsbombpy mplsoccer soccerdata xgboost scikit-learn seaborn

In [2]:
# --- Notebook 01: Modern Era Multi-League Data Ingestion ---
from statsbombpy import sb
import pandas as pd
import os

# 1. Setup Directory
if not os.path.exists('data'):
    os.makedirs('data')

print("Fetching competition metadata...")
competitions = sb.competitions()

# Target leagues for maximum volume and quality
target_leagues = [
    'Premier League',
    'La Liga',
    '1. Bundesliga',
    'Serie A',
    'Ligue 1',
    'Champions League',
    'FIFA World Cup',
    'UEFA Euro'
]

# 2. Filter for Target Leagues AND Seasons from 2015/16 onwards
# We extract the start year from the 'season_name' string (e.g., '2019/2020' -> 2019)
def get_start_year(season_name):
    try:
        # Handles '2019/2020' or just '2022'
        return int(season_name.split('/')[0])
    except:
        return 0

competitions['start_year'] = competitions['season_name'].apply(get_start_year)

# Filter: League must be in list AND start year must be >= 2015
selected_comps = competitions[
    (competitions['competition_name'].isin(target_leagues)) &
    (competitions['start_year'] >= 2015)
].sort_values(['competition_name', 'start_year'], ascending=[True, False])

all_match_shots = []

print(f"Starting ingestion for {len(selected_comps)} modern seasons...")

# 3. Loop through each available competition/season pair
for index, row in selected_comps.iterrows():
    c_id = row['competition_id']
    s_id = row['season_id']
    c_name = row['competition_name']
    s_name = row['season_name']

    print(f"Processing: {c_name} | {s_name}...")

    try:
        matches = sb.matches(competition_id=c_id, season_id=s_id)
        match_list = matches['match_id'].tolist()

        for m_id in match_list:
            # Get only the shots to save memory and time
            events = sb.events(match_id=m_id, split=True)

            if 'shots' in events:
                shots_df = events['shots']
                # Metadata tagging
                shots_df['match_id'] = m_id
                shots_df['competition_name'] = c_name
                shots_df['season_name'] = s_name
                all_match_shots.append(shots_df)
    except Exception as e:
        print(f"Skipping {c_name} {s_name} due to match error.")

# 4. Consolidate and Save
if all_match_shots:
    final_df = pd.concat(all_match_shots, ignore_index=True)

    # Final cleanup: Remove any duplicates if they exist
    final_df = final_df.drop_duplicates(subset=['id'])

    output_path = 'data/01_raw_modern_master_shots.csv'
    final_df.to_csv(output_path, index=False)

    print("\n" + "="*40)
    print("MODERN ERA INGESTION COMPLETE")
    print(f"Total Shots Collected: {len(final_df)}")
    print(f"Oldest Season Included: {final_df['season_name'].min()}")
    print(f"File Saved to: {output_path}")
    print("="*40)
else:
    print("No data found matching criteria.")

Fetching competition metadata...
Starting ingestion for 21 modern seasons...
Processing: 1. Bundesliga | 2023/2024...


/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: 

Processing: 1. Bundesliga | 2015/2016...


/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: 

Processing: Champions League | 2018/2019...


...

Processing: UEFA Euro | 2024...


/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: 

Processing: UEFA Euro | 2020...


/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: 


MODERN ERA INGESTION COMPLETE
Total Shots Collected: 58465
Oldest Season Included: 2015/2016
File Saved to: data/01_raw_modern_master_shots.csv


In [4]:
from statsbombpy import sb
import pandas as pd

# Fetch all free competitions
comps = sb.competitions()

# Updated target list to include Tournaments
target_list = [
    'Premier League', 'La Liga', '1. Bundesliga', 'Serie A', 'Ligue 1',
    'Champions League', 'FIFA World Cup', 'UEFA Euro'
]

# Filter for the modern era (2015 onwards)
available_modern = comps[
    (comps['competition_name'].isin(target_list)) &
    (comps['season_name'].str.contains('2015|2016|2017|2018|2019|2020|2021|2022|2023|2024|2025|2026'))
]

print("--- MODERN TOURNAMENTS & LEAGUES (FREE) ---")
# Sorting for easier reading
print(available_modern[['competition_name', 'season_name', 'competition_id', 'season_id']].sort_values(['competition_name', 'season_name']))

--- MODERN TOURNAMENTS & LEAGUES (FREE) ---
    competition_name season_name  competition_id  season_id
1      1. Bundesliga   2015/2016               9         27
0      1. Bundesliga   2023/2024               9        281
7   Champions League   2014/2015              16         26
6   Champions League   2015/2016              16         27
5   Champions League   2016/2017              16          2
4   Champions League   2017/2018              16          1
3   Champions League   2018/2019              16          4
30    FIFA World Cup        2018              43          3
29    FIFA World Cup        2022              43        106
44           La Liga   2014/2015              11         26
43           La Liga   2015/2016              11         27
42           La Liga   2016/2017              11          2
41           La Liga   2017/2018              11          1
40           La Liga   2018/2019              11          4
39           La Liga   2019/2020              11        

/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [5]:
from statsbombpy import sb
import pandas as pd
import os

# Load your existing data to see what seasons are already there
# We use this to prevent duplicates
existing_df = pd.read_csv('data/01_raw_modern_master_shots.csv')
already_collected = existing_df[['competition_name', 'season_name']].drop_duplicates()

# Get all available free competitions
comps = sb.competitions()

# Men's target list
men_leagues = ['Premier League', 'La Liga', '1. Bundesliga', 'Serie A', 'Ligue 1', 'Champions League', 'FIFA World Cup', 'UEFA Euro']

def get_start_year(name):
    try: return int(name.split('/')[0])
    except: return int(name) if name.isdigit() else 0

comps['start_year'] = comps['season_name'].apply(get_start_year)

# Filter for Men's data from 2010 onwards
all_potential = comps[
    (comps['competition_name'].isin(men_leagues)) &
    (comps['start_year'] >= 2010)
]

# Identify only the NEW seasons (those not in your existing_df)
new_seasons = pd.merge(all_potential, already_collected,
                       on=['competition_name', 'season_name'],
                       how='left', indicator=True)
new_seasons = new_seasons[new_seasons['_merge'] == 'left_only']

print(f"New seasons to collect: {len(new_seasons)}")
print(new_seasons[['competition_name', 'season_name']])

New seasons to collect: 10
    competition_name season_name
6   Champions League   2014/2015
7   Champions League   2013/2014
8   Champions League   2012/2013
9   Champions League   2011/2012
10  Champions League   2010/2011
19           La Liga   2014/2015
20           La Liga   2013/2014
21           La Liga   2012/2013
22           La Liga   2011/2012
23           La Liga   2010/2011


/usr/local/lib/python3.12/dist-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


In [6]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='statsbombpy')

all_new_shots = []

# Using the 'new_seasons' dataframe we generated in the audit
print(f"Starting incremental pull for {len(new_seasons)} elite seasons (2010-2015)...")

for i, (index, row) in enumerate(new_seasons.iterrows()):
    c_id, s_id = row['competition_id'], row['season_id']
    c_name, s_name = row['competition_name'], row['season_name']

    print(f"[{i+1}/{len(new_seasons)}] Adding: {c_name} ({s_name})...")

    try:
        matches = sb.matches(competition_id=c_id, season_id=s_id)
        for m_id in matches['match_id']:
            # split=True ensures we only pull the 'shots' table
            events = sb.events(match_id=m_id, split=True)
            if 'shots' in events:
                shots = events['shots']
                shots['competition_name'] = c_name
                shots['season_name'] = s_name
                all_new_shots.append(shots)
    except Exception as e:
        print(f"Error on {c_name} {s_name}: {e}")
        continue

# Append to your master file
if all_new_shots:
    new_df = pd.concat(all_new_shots, ignore_index=True)
    # Load the 55k you already saved
    existing_df = pd.read_csv('data/01_raw_modern_master_shots.csv')

    # Merge and deduplicate
    updated_master = pd.concat([existing_df, new_df], ignore_index=True)
    updated_master = updated_master.drop_duplicates(subset=['id'])

    # Save the new total
    updated_master.to_csv('data/01_raw_modern_master_shots.csv', index=False)

    print("\n" + "="*40)
    print("INCREMENTAL UPDATE COMPLETE")
    print(f"Final Shot Count: {len(updated_master)}")
    print(f"New Data Added: {len(new_df)} shots")
    print("="*40)
else:
    print("No new data was collected.")


Starting incremental pull for 10 elite seasons (2010-2015)...
[1/10] Adding: Champions League (2014/2015)...
[2/10] Adding: Champions League (2013/2014)...
[3/10] Adding: Champions League (2012/2013)...
[4/10] Adding: Champions League (2011/2012)...
[5/10] Adding: Champions League (2010/2011)...
[6/10] Adding: La Liga (2014/2015)...
[7/10] Adding: La Liga (2013/2014)...
[8/10] Adding: La Liga (2012/2013)...
[9/10] Adding: La Liga (2011/2012)...
[10/10] Adding: La Liga (2010/2011)...

INCREMENTAL UPDATE COMPLETE
Final Shot Count: 62718
New Data Added: 4253 shots


In [8]:
import pandas as pd

file_path = 'data/01_raw_modern_master_shots.csv'

try:
    df_existing = pd.read_csv(file_path)

    # Check which columns actually exist to avoid the error
    available_cols = df_existing.columns.tolist()
    print(f"Columns found in CSV: {available_cols[:10]}...") # Printing first 10 for brevity

    # Use names instead of IDs for the audit, as these are more reliable in your current CSV
    banked_seasons = df_existing[['competition_name', 'season_name']].drop_duplicates()

    print(f"\n--- CURRENTLY BANKED DATA ({len(df_existing)} shots) ---")
    print(banked_seasons.sort_values(['competition_name', 'season_name']))

    # Create the set for comparison in Script 2
    banked_names = set(zip(banked_seasons['competition_name'], banked_seasons['season_name']))

except FileNotFoundError:
    print("Master file not found. Starting from scratch.")
    banked_names = set()
except KeyError as e:
    print(f"Missing columns: {e}")
    # If even names are missing, we might need to check your CSV structure
    print("Suggested fix: Check the first few rows of your dataframe using df_existing.head()")

Columns found in CSV: ['id', 'index', 'period', 'timestamp', 'minute', 'second', 'type', 'possession', 'possession_team', 'play_pattern']...

--- CURRENTLY BANKED DATA (62718 shots) ---
       competition_name season_name
916       1. Bundesliga   2015/2016
0         1. Bundesliga   2023/2024
58618  Champions League   2010/2011
58559  Champions League   2011/2012
58529  Champions League   2012/2013
58497  Champions League   2013/2014
58465  Champions League   2014/2015
8832   Champions League   2015/2016
8805   Champions League   2016/2017
8777   Champions League   2017/2018
8747   Champions League   2018/2019
10379    FIFA World Cup        2018
8885     FIFA World Cup        2022
61979           La Liga   2010/2011
61106           La Liga   2011/2012
60385           La Liga   2012/2013
59575           La Liga   2013/2014
58644           La Liga   2014/2015
16416           La Liga   2015/2016
15534           La Liga   2016/2017
14563           La Liga   2017/2018
13676           La Lig

In [9]:
from statsbombpy import sb

# 1. Fetch all free competitions
all_comps = sb.competitions()

# 2. Filter for Men's football and 2010+
def get_year(name):
    try: return int(name.split('/')[0])
    except: return int(name) if name.isdigit() else 0

all_comps['year'] = all_comps['season_name'].apply(get_year)
available_men = all_comps[(all_comps['competition_gender'] == 'male') & (all_comps['year'] >= 2010)].copy()

# 3. Compare using names
new_seasons_to_get = available_men[
    ~available_men.apply(lambda x: (x['competition_name'], x['season_name']) in banked_names, axis=1)
]

print(f"--- NEW SEASONS TO BE COLLECTED ({len(new_seasons_to_get)}) ---")
if not new_seasons_to_get.empty:
    # Sort to see the most relevant ones
    print(new_seasons_to_get[['competition_name', 'season_name', 'competition_id', 'season_id']].sort_values('competition_name'))
else:
    print("No new men's data found.")

--- NEW SEASONS TO BE COLLECTED (4) ---
          competition_name season_name  competition_id  season_id
2   African Cup of Nations        2023            1267        107
21            Copa America        2024             223        282
37     Indian Super league   2021/2022            1238        108
61     Major League Soccer        2023              44        107


In [10]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module='statsbombpy')

# 1. Filter the 'new_seasons_to_get' to exclude ISL
final_targets = new_seasons_to_get[new_seasons_to_get['competition_name'] != 'Indian Super league']

if not final_targets.empty:
    all_new_shots = []

    print(f"Starting ingestion for {len(final_targets)} elite international/club seasons...")

    for i, (index, row) in enumerate(final_targets.iterrows()):
        c_id, s_id = row['competition_id'], row['season_id']
        c_name, s_name = row['competition_name'], row['season_name']

        print(f"[{i+1}/{len(final_targets)}] Fetching: {c_name} {s_name}...")

        try:
            matches = sb.matches(competition_id=c_id, season_id=s_id)
            for m_id in matches['match_id']:
                # split=True pulls only the shots table
                events = sb.events(match_id=m_id, split=True)
                if 'shots' in events:
                    shots = events['shots']
                    # Attaching metadata for the master file
                    shots['competition_name'] = c_name
                    shots['season_name'] = s_name
                    all_new_shots.append(shots)
        except Exception as e:
            print(f"Error skipping {c_name}: {e}")
            continue

    # 2. Merge with Existing Master
    if all_new_shots:
        new_shots_df = pd.concat(all_new_shots, ignore_index=True)

        # Load your existing 62.7k records
        df_existing = pd.read_csv('data/01_raw_modern_master_shots.csv')

        # Combine
        updated_master = pd.concat([df_existing, new_shots_df], ignore_index=True)

        # Final safety check for duplicates based on StatsBomb's unique Event ID
        updated_master = updated_master.drop_duplicates(subset=['id'])

        # Overwrite the master file
        updated_master.to_csv('data/01_raw_modern_master_shots.csv', index=False)

        print("\n" + "="*40)
        print("COLLECTION COMPLETE")
        print(f"Old count: {len(df_existing)}")
        print(f"New total count: {len(updated_master)}")
        print(f"Leagues added: {final_targets['competition_name'].tolist()}")
        print("="*40)
else:
    print("No target seasons identified for ingestion.")

Starting ingestion for 3 elite international/club seasons...
[1/3] Fetching: African Cup of Nations 2023...
[2/3] Fetching: Copa America 2024...
[3/3] Fetching: Major League Soccer 2023...

COLLECTION COMPLETE
Old count: 62718
New total count: 64898
Leagues added: ['African Cup of Nations', 'Copa America', 'Major League Soccer']


In [11]:
import pandas as pd

# Load your existing master data
file_path = 'data/01_raw_modern_master_shots.csv'

try:
    df_existing = pd.read_csv(file_path)

    # We identify what's banked by looking at the combination of names
    # This avoids the ID KeyError we saw earlier
    banked_seasons = df_existing[['competition_name', 'season_name']].drop_duplicates()

    print(f"--- CURRENTLY BANKED DATA ({len(df_existing)} shots) ---")
    print(banked_seasons.sort_values(['competition_name', 'season_name']))

    # Create a lookup set
    banked_set = set(zip(banked_seasons['competition_name'], banked_seasons['season_name']))
except Exception as e:
    print(f"Error loading file: {e}")
    banked_set = set()

--- CURRENTLY BANKED DATA (64898 shots) ---
             competition_name season_name
916             1. Bundesliga   2015/2016
0               1. Bundesliga   2023/2024
62718  African Cup of Nations        2023
58618        Champions League   2010/2011
58559        Champions League   2011/2012
58529        Champions League   2012/2013
58497        Champions League   2013/2014
58465        Champions League   2014/2015
8832         Champions League   2015/2016
8805         Champions League   2016/2017
8777         Champions League   2017/2018
8747         Champions League   2018/2019
63962            Copa America        2024
10379          FIFA World Cup        2018
8885           FIFA World Cup        2022
61979                 La Liga   2010/2011
61106                 La Liga   2011/2012
60385                 La Liga   2012/2013
59575                 La Liga   2013/2014
58644                 La Liga   2014/2015
16416                 La Liga   2015/2016
15534                 La Liga   

In [12]:
from statsbombpy import sb

# 1. Fetch ALL available free competitions
all_available = sb.competitions()

# 2. Filter for Men's football only
# Note: We aren't filtering by year yet, so you can see EVERYTHING available.
men_only = all_available[all_available['competition_gender'] == 'male'].copy()

# 3. Find the missing pieces
missing_seasons = men_only[
    ~men_only.apply(lambda x: (x['competition_name'], x['season_name']) in banked_set, axis=1)
]

print(f"--- MISSING MEN'S SEASONS ({len(missing_seasons)}) ---")
if not missing_seasons.empty:
    # We display them so you can decide which ones to keep
    print(missing_seasons[['competition_name', 'season_name', 'competition_id', 'season_id']].sort_values('competition_name'))
else:
    print("You have collected every single free Men's match available on StatsBomb!")

--- MISSING MEN'S SEASONS (33) ---
         competition_name season_name  competition_id  season_id
12       Champions League   2009/2010              16         21
13       Champions League   2008/2009              16         41
14       Champions League   2006/2007              16         39
15       Champions League   2004/2005              16         37
16       Champions League   2003/2004              16         44
17       Champions League   1999/2000              16         76
18       Champions League   1972/1973              16        277
19       Champions League   1971/1972              16         71
20       Champions League   1970/1971              16        276
22           Copa del Rey   1983/1984              87         84
23           Copa del Rey   1982/1983              87        268
24           Copa del Rey   1977/1978              87        279
28     FIFA U20 World Cup        1979            1470        274
36         FIFA World Cup        1958              43  

In [13]:
import pandas as pd

# Load the existing master data
file_path = 'data/01_raw_modern_master_shots.csv'

try:
    df = pd.read_csv(file_path)
    total_count = len(df)

    # Check for the statsbomb xG column
    col_name = 'shot_statsbomb_xg'
    if col_name in df.columns:
        xg_count = df[col_name].notna().sum()
        print(f"Total records: {total_count}")
        print(f"Records with '{col_name}': {xg_count}")
        print(f"Percentage: {round((xg_count / total_count) * 100, 2)}%")
    else:
        print(f"Column '{col_name}' does not exist in the dataset.")
        # Let's list columns that might be related
        xg_cols = [c for c in df.columns if 'xg' in c.lower()]
        if xg_cols:
            print(f"Found other potential xG columns: {xg_cols}")
            for col in xg_cols:
                count = df[col].notna().sum()
                print(f" - {col}: {count}")
except Exception as e:
    print(f"Error: {e}")


Total records: 64898
Records with 'shot_statsbomb_xg': 64898
Percentage: 100.0%


In [14]:
from statsbombpy import sb
import pandas as pd
import warnings

# Suppress the NoAuthWarning for cleaner output
warnings.filterwarnings("ignore", category=UserWarning, module='statsbombpy')

# List of specific potential "Hidden" modern targets to check
test_targets = [
    {'name': 'MLS 2018', 'c_id': 44, 's_id': 1},
    {'name': 'MLS 2017', 'c_id': 44, 's_id': 24},
    {'name': 'Copa America 2021', 'c_id': 11, 's_id': 90}
]

print("--- xG AVAILABILITY CHECK (Hidden Modern Data) ---")
valid_targets = []

for target in test_targets:
    try:
        # 1. Get matches for the competition
        matches = sb.matches(competition_id=target['c_id'], season_id=target['s_id'])

        if matches.empty:
            print(f"{target['name']}: ⚠️ No matches found in Open Data.")
            continue

        # 2. Grab a sample match ID
        sample_match = matches['match_id'].iloc[0]

        # 3. Pull events for that match
        events = sb.events(match_id=sample_match, split=True)

        if 'shots' in events:
            shots = events['shots']
            has_xg_col = 'shot_statsbomb_xg' in shots.columns

            if has_xg_col:
                valid_xg_count = shots['shot_statsbomb_xg'].notna().sum()
                total_shots = len(shots)
                coverage = (valid_xg_count / total_shots) * 100

                if coverage > 90:
                    status = f"✅ {round(coverage, 1)}% xG coverage"
                    valid_targets.append(target)
                else:
                    status = f"❌ Only {round(coverage, 1)}% coverage (Likely incomplete)"
            else:
                status = "❌ No 'shot_statsbomb_xg' column found"

            print(f"{target['name']}: {status}")

    except Exception as e:
        print(f"{target['name']}: ⚠️ Error: {str(e)}")

print("\n" + "="*30)
if valid_targets:
    print("READY FOR INGESTION:")
    for vt in valid_targets:
        print(f"- {vt['name']} (Comp: {vt['c_id']}, Season: {vt['s_id']})")
else:
    print("No additional 100% xG seasons found in these IDs.")
print("="*30)

--- xG AVAILABILITY CHECK (Hidden Modern Data) ---
MLS 2018: ⚠️ Error: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/statsbomb/open-data/master/data/matches/44/1.json
MLS 2017: ⚠️ Error: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/statsbomb/open-data/master/data/matches/44/24.json
Copa America 2021: ✅ 100.0% xG coverage

READY FOR INGESTION:
- Copa America 2021 (Comp: 11, Season: 90)


In [15]:
import warnings
import pandas as pd
from statsbombpy import sb

warnings.filterwarnings("ignore", category=UserWarning, module='statsbombpy')

# Target configuration from your successful test
target_c_id = 11
target_s_id = 90
target_name = "Copa America 2021"

print(f"Starting ingestion for {target_name}...")

new_shots = []

try:
    # 1. Fetch matches
    matches = sb.matches(competition_id=target_c_id, season_id=target_s_id)
    total_matches = len(matches)
    print(f"Found {total_matches} matches to process.")

    # 2. Loop through matches and pull shots
    for i, m_id in enumerate(matches['match_id']):
        if (i + 1) % 5 == 0 or (i + 1) == total_matches:
            print(f"Progress: [{i+1}/{total_matches}] matches processed...")

        events = sb.events(match_id=m_id, split=True)
        if 'shots' in events:
            s_df = events['shots']
            s_df['competition_name'] = "Copa America"
            s_df['season_name'] = "2021"
            new_shots.append(s_df)

    # 3. Combine and Merge
    if new_shots:
        new_data_df = pd.concat(new_shots, ignore_index=True)

        # Load your existing master file
        df_existing = pd.read_csv('data/01_raw_modern_master_shots.csv')

        # Combine and deduplicate based on StatsBomb's unique 'id'
        updated_master = pd.concat([df_existing, new_data_df], ignore_index=True)
        updated_master = updated_master.drop_duplicates(subset=['id'])

        # Save back to CSV
        updated_master.to_csv('data/01_raw_modern_master_shots.csv', index=False)

        print("\n" + "="*40)
        print("COPA AMÉRICA INGESTION COMPLETE")
        print(f"New shots added: {len(new_data_df)}")
        print(f"NEW TOTAL SHOT COUNT: {len(updated_master)}")
        print("="*40)
    else:
        print("No shots found in the retrieved matches.")

except Exception as e:
    print(f"An error occurred during ingestion: {e}")

Starting ingestion for Copa America 2021...
Found 35 matches to process.
Progress: [5/35] matches processed...
Progress: [10/35] matches processed...
Progress: [15/35] matches processed...
Progress: [20/35] matches processed...
Progress: [25/35] matches processed...
Progress: [30/35] matches processed...
Progress: [35/35] matches processed...

COPA AMÉRICA INGESTION COMPLETE
New shots added: 839
NEW TOTAL SHOT COUNT: 64898
